In [11]:
import warnings
warnings.filterwarnings("ignore")

In [12]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.optim import AdamW

In [13]:
sentences = [
    ["John", "works", "at", "Google", "in", "California"],
    ["Mary", "is", "a", "doctor"],
]

labels = [
    ["NOUN", "VERB", "ADP", "ORG", "ADP", "LOC"],
    ["NOUN", "VERB", "DET", "NOUN"]
]

label_list = ["NOUN", "VERB", "ADP", "DET", "ORG", "LOC"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [14]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
def tokenize_and_align(sentences, labels):
    encodings = tokenizer(
        sentences,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    all_labels = []

    for i, label in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=i)
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[label[word_idx]])

        all_labels.append(label_ids)

    encodings["labels"] = torch.tensor(all_labels)
    return encodings

data = tokenize_and_align(sentences, labels)

In [16]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)

model.train()

for epoch in range(2):
    optimizer.zero_grad()
    
    outputs = model(
        input_ids=data["input_ids"],
        attention_mask=data["attention_mask"],
        labels=data["labels"]
    )
    
    loss = outputs.loss
    print("Epoch:", epoch+1, "Loss:", loss.item())
    
    loss.backward()
    optimizer.step()

Epoch: 1 Loss: 1.857692003250122
Epoch: 2 Loss: 1.4415433406829834


In [9]:
from seqeval.metrics import classification_report, f1_score

model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=data["input_ids"],
        attention_mask=data["attention_mask"]
    )

predictions = torch.argmax(outputs.logits, dim=2)

true_labels = []
pred_labels = []

for i in range(len(labels)):
    true = []
    pred = []
    
    word_ids = tokenizer(sentences[i], is_split_into_words=True).word_ids()
    
    for j, word_idx in enumerate(word_ids):
        if word_idx is not None:
            true.append(labels[i][word_idx])
            pred.append(id2label[predictions[i][j].item()])
    
    true_labels.append(true)
    pred_labels.append(pred)

print("F1 Score:", f1_score(true_labels, pred_labels))
print(classification_report(true_labels, pred_labels))

F1 Score: 1.0
              precision    recall  f1-score   support

          DP       1.00      1.00      1.00         2
         ERB       1.00      1.00      1.00         2
          ET       1.00      1.00      1.00         1
          OC       1.00      1.00      1.00         1
         OUN       1.00      1.00      1.00         3

   micro avg       1.00      1.00      1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



In [10]:
sentence = ["John", "works", "at", "Google", "in", "California"]

inputs = tokenizer(sentence, return_tensors="pt", is_split_into_words=True)

with torch.no_grad():
    outputs = model(**inputs)

preds = torch.argmax(outputs.logits, dim=2)

result = []
for i, token in enumerate(sentence):
    result.append((token, id2label[preds[0][i].item()]))

print(result)

[('John', 'ADP'), ('works', 'NOUN'), ('at', 'VERB'), ('Google', 'ADP'), ('in', 'ORG'), ('California', 'ADP')]
